In [ ]:
import re, os, psycopg2, datetime,importlib
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from dotenv import load_dotenv
import segmentação.inicializador as init

importlib.reload(init)

load_dotenv()

host = os.getenv("HOST")
database = os.getenv("DATABASE")
user= os.getenv("USER")
password= os.getenv("PASSWORD")

service, options = init.start_driver()

# Inicia-se a instância do Chrome WebDriver com as definidas 'options' e 'service', basicamente o driver É o google chrome.
driver = init.go_to_site(service, options, 'https://www.bcb.gov.br/controleinflacao/historicotaxasjuros')

tab_txa_selic = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "historicotaxasjuros")))

tbody_tab_txa_selic = WebDriverWait(tab_txa_selic, 10).until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

tr_tab_txa_selic = WebDriverWait(tbody_tab_txa_selic, 10).until(EC.presence_of_all_elements_located((By.TAG_NAME, "tr")))

reuniao, data_reuniao, extraordinaria, vies, inicio_vigencia, fim_vigencia, taxa_selic, selic_acumulada_periodo, selic_media_anual = [], [], [], [], [], [], [], [], []

for count, _ in enumerate(tr_tab_txa_selic):

    info_selic = tr_tab_txa_selic[count].text.split()
        
    try:
        info_selic.remove("-")
    except ValueError:
        pass
    x, y = re.findall(r"\(\d+\)", info_selic[1]), re.findall(r"\(\d+\)", info_selic[2]) 
    if x != []:
        info_selic.remove(x[0])
    elif y != []:
        info_selic.remove(y[0])
    
    is_ex = False
    if info_selic[1] == 'ex.':
        is_ex = True
        info_selic.remove('ex.')

    if info_selic[2] == 'sem':
        info_selic[2] = info_selic[2] + ' ' + info_selic[3]
        info_selic.remove(info_selic[3])

    if len(info_selic) <= 6:
        reuniao.append(info_selic[0])
        data_reuniao.append(info_selic[1])
        extraordinaria.append(is_ex)
        vies.append(info_selic[2])
        inicio_vigencia.append(info_selic[3])
        fim_vigencia.append(None)
        taxa_selic.append(info_selic[4])
        #tban.append(info_selic[5])
        selic_acumulada_periodo.append(None)
        selic_media_anual.append(None)
    else:
        reuniao.append(info_selic[0])
        data_reuniao.append(info_selic[1])
        extraordinaria.append(is_ex)
        vies.append(info_selic[2])
        inicio_vigencia.append(info_selic[3])
        fim_vigencia.append(info_selic[4])
        taxa_selic.append(info_selic[5])
        #tban.append(info_selic[6])
        selic_acumulada_periodo.append(info_selic[7])
        selic_media_anual.append(info_selic[8])
    
dict_selic = {}

dict_selic['reuniao'] = reuniao
dict_selic['data_reuniao'] = data_reuniao
dict_selic['extraordinaria'] = extraordinaria
dict_selic['vies'] = vies
dict_selic['inicio_vigencia'] = inicio_vigencia
dict_selic['fim_vigencia'] = fim_vigencia
dict_selic['taxa_selic'] = taxa_selic
dict_selic['selic_acumulada_periodo'] = selic_acumulada_periodo
dict_selic['selic_media_anual'] = selic_media_anual

selic = dict_selic

def parse_float(value):
    try:
        return float(value.replace(',', '.'))
    except ValueError:
        return None
    except AttributeError:
        return None

def change_data_format(value):
    try:
        data = value.replace("/", '-')
        data = datetime.datetime.strptime(data, "%d-%m-%Y").strftime("%Y-%m-%d")
        return data
    except AttributeError:
        return value
    except ValueError:
        return value

def clear_na(value):
    try:
        return value.replace('n/a', 'não aplicável')
    except AttributeError:
        return value
    
lista_to_insert = [
    [
    valores[0],
    change_data_format(valores[1]),
    valores[2],
    clear_na(valores[3]),
    change_data_format(valores[4]),
    change_data_format(valores[5]),
    parse_float(valores[6]),
    parse_float(valores[7]),
    parse_float(valores[8])
    ]
    for valores in zip(
        dict_selic['reuniao'],
        dict_selic['data_reuniao'],
        dict_selic['extraordinaria'],
        dict_selic['vies'],
        dict_selic['inicio_vigencia'],
        dict_selic['fim_vigencia'],
        dict_selic['taxa_selic'],
        dict_selic['selic_acumulada_periodo'],
        dict_selic['selic_media_anual']
    )
]

try:
    # Conectar ao banco de dados
    #conn = psycopg2.connect(host=host, database=database, user=user, password=password, sslmode='require')
    conn = psycopg2.connect(host=host, database=database, user=user, password=password)
    cursor = conn.cursor()

    # Criar o banco de dados, caso não exista
    #cursor.execute("CREATE DATABASE IF NOT EXISTS indices_financeiros;")

    # Criar tabela para armazenar os índices
    create_table_query = """
    CREATE TABLE IF NOT EXISTS selic (
        reuniao VARCHAR(10) NOT NULL,
        data_reuniao DATE NOT NULL PRIMARY KEY,
        extraordinaria BOOLEAN NOT NULL,
        vies VARCHAR(20) NOT NULL,
        inicio_vigencia DATE NOT NULL,
        fim_vigencia DATE,
        taxa_selic NUMERIC NOT NULL,
        selic_acumulada_periodo NUMERIC,
        selic_media_anual NUMERIC
    );
    """
    cursor.execute(create_table_query)

    # Inserir os dados na tabela
    insert_query = """
    INSERT INTO selic (reuniao, data_reuniao, extraordinaria, vies, inicio_vigencia, fim_vigencia, taxa_selic, selic_acumulada_periodo, selic_media_anual)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (data_reuniao) DO UPDATE
    SET reuniao = EXCLUDED.reuniao,
    data_reuniao = EXCLUDED.data_reuniao,
    extraordinaria = EXCLUDED.extraordinaria,
    vies = EXCLUDED.vies,
    inicio_vigencia = EXCLUDED.inicio_vigencia,
    fim_vigencia = EXCLUDED.fim_vigencia,
    taxa_selic = EXCLUDED.taxa_selic,
    selic_acumulada_periodo = EXCLUDED.selic_acumulada_periodo,
    selic_media_anual = EXCLUDED.selic_media_anual;
    """
    cursor.executemany(insert_query, lista_to_insert)

    # Confirmar as alterações no banco de dados
    conn.commit()
    print("Dados inseridos com sucesso!")

except psycopg2.Error as e:
    print(f"Erro ao conectar ou manipular o banco de dados: {e}")
finally:
    if conn:
        cursor.close()
        conn.close()

driver.quit()

The chromedriver version (133.0.6943.53) detected in PATH at C:\Windows\chromedriver.exe might not be compatible with the detected chrome version (134.0.6998.89); currently, chromedriver 134.0.6998.88 is recommended for chrome 134.*, so it is advised to delete the driver in PATH and retry


268ª
267ª
266ª
265ª
264ª
263ª
262ª
261ª
260ª
259ª
258ª
257ª
256ª
255ª
254ª
253ª
252ª
251ª
250ª
249ª
248ª
247ª
246ª
245ª
244ª
243ª
242ª
241ª
240ª
239ª
238ª
237ª
236ª
235ª
234ª
233ª
232ª
231ª
230ª
229ª
228ª
227ª
226ª
225ª
224ª
223ª
222ª
221ª
220ª
219ª
218ª
217ª
216ª
215ª
214ª
213ª
212ª
211ª
210ª
209ª
208ª
207ª
206ª
205ª
204ª
203ª
202ª
201ª
200ª
199ª
198ª
197ª
196ª
195ª
194ª
193ª
192ª
191ª
190ª
189ª
188ª
187ª
186ª
185ª
184ª
183ª
182ª
181ª
180ª
179ª
178ª
177ª
176ª
175ª
174ª
173ª
172ª
171ª
170ª
169ª
168ª
167ª
166ª
165ª
164ª
163ª
162ª
161ª
160ª
159ª
158ª
157ª
156ª
155ª
154ª
153ª
152ª
151ª
150ª
149ª
148ª
147ª
146ª
145ª
144ª
143ª
142ª
141ª
140ª
139ª
138ª
137ª
136ª
135ª
134ª
133ª
132ª
131ª
130ª
129ª
128ª
127ª
126ª
125ª
124ª
123ª
122ª
121ª
120ª
119ª
118ª
117ª
116ª
115ª
114ª
113ª
112ª
111ª
110ª
109ª
108ª
107ª
106ª
105ª
104ª
103ª
102ª
101ª
100ª
99ª
98ª
97ª
96ª
95ª
94ª
93ª
92ª
91ª
90ª
89ª
88ª
87ª
86ª
85ª
84ª
83ª
82ª
81ª
80ª
79ª
78ª
77ª
76ª
75ª
74ª
73ª
72ª
71ª
70ª
69ª
68ª
67ª
66ª
65ª
64ª
63ª
62ª
61ª

In [2]:
import psycopg2

host="ep-shiny-violet-a5x5bctb-pooler.us-east-2.aws.neon.tech"
database="neondb"
user="neondb_owner"
password="ZgiIsd9BHSJ4"

try:
    conn = psycopg2.connect(
        host=host,
        database=database,
        user=user,
        password=password,
        sslmode="require"
    )
    print("Conexão bem-sucedida!")
    conn.close()
except Exception as e:
    print(f"Erro ao conectar ao banco: {e}")


Erro ao conectar ao banco: connection to server at "ep-shiny-violet-a5x5bctb.us-east-2.aws.neon.tech" (3.23.186.13), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "ep-shiny-violet-a5x5bctb.us-east-2.aws.neon.tech" (3.143.47.40), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "ep-shiny-violet-a5x5bctb.us-east-2.aws.neon.tech" (3.131.64.200), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?

